In [2]:
import pandas as pd
import numpy as np
import datatable as dt
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit,StratifiedKFold, GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from statistics import mean
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [3]:
phenotype_df = pd.read_csv('/home/student2024/Desktop/Project_Depression/combined_dataset_phenotypes_corrected.csv')
phenotype_df.head()

,ID,Sex,Age,Depression,Study,depr_one_hot
0,GSM3562834_3999840035_R01C01,0,73,Control,1,0
1,GSM3562835_3999840035_R01C02,1,32,Case,1,1
2,GSM3562836_3999840035_R02C01,0,51,Case,1,1
3,GSM3562837_3999840035_R02C02,1,30,Case,1,1
4,GSM3562838_3999840035_R03C01,0,53,Case,1,1


In [4]:
mval = dt.fread('/home/student2024/Desktop/Project_Depression/Combined_Mval_non_adj.csv')
mval = mval.to_pandas()
#mval = mval.rename(columns={'C0':'ID'})
mval = mval.T
mval.columns = mval.iloc[0]
mval = mval[1:]
mval.head()

C0,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,cg00008004,...,cg27612019,cg27614432,cg27634071,cg27634744,cg27648216,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648
GSM3562834_3999840035_R01C01,3.699859,-4.60819,-5.758384,-3.428361,-1.607677,-6.670791,-3.107789,5.820279,-5.475742,-3.163562,...,0.402738,-4.935797,-5.004572,3.629387,3.625561,3.712343,2.162832,-3.205565,-4.658252,1.941852
GSM3562835_3999840035_R01C02,3.829019,-5.517733,-5.364797,-3.806508,-3.06947,-6.675133,-2.910615,6.075227,-5.647746,-2.860776,...,0.730073,-4.556461,-4.425977,3.305114,3.534774,3.963833,3.127694,-3.446988,-4.824407,1.924064
GSM3562836_3999840035_R02C01,4.543859,-5.298929,-5.672585,-3.913202,-2.032083,-6.644306,-4.005748,6.224032,-5.629312,-3.375355,...,0.916134,-4.753392,-5.297943,3.110546,4.186899,3.808285,3.060312,-2.811014,-4.545404,2.232796
GSM3562837_3999840035_R02C02,3.935193,-4.879178,-5.518779,-3.143026,-1.970269,-6.487348,-3.460816,5.762759,-5.47399,-3.410145,...,0.241682,-4.635074,-4.702293,2.967417,3.834561,3.631687,3.381633,-3.322698,-4.960753,2.02318
GSM3562838_3999840035_R03C01,4.207299,-4.579331,-5.579593,-3.098945,-1.041133,-6.402856,-2.857366,5.775951,-5.572881,-2.750484,...,0.428073,-5.145249,-5.329275,2.833336,3.631742,3.819408,3.013578,-3.525631,-4.398742,2.936999


In [5]:
mval.index.name = 'ID'
mval.reset_index(inplace=True)
mval.head()

C0,ID,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,...,cg27612019,cg27614432,cg27634071,cg27634744,cg27648216,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648
0,GSM3562834_3999840035_R01C01,3.699859,-4.60819,-5.758384,-3.428361,-1.607677,-6.670791,-3.107789,5.820279,-5.475742,...,0.402738,-4.935797,-5.004572,3.629387,3.625561,3.712343,2.162832,-3.205565,-4.658252,1.941852
1,GSM3562835_3999840035_R01C02,3.829019,-5.517733,-5.364797,-3.806508,-3.06947,-6.675133,-2.910615,6.075227,-5.647746,...,0.730073,-4.556461,-4.425977,3.305114,3.534774,3.963833,3.127694,-3.446988,-4.824407,1.924064
2,GSM3562836_3999840035_R02C01,4.543859,-5.298929,-5.672585,-3.913202,-2.032083,-6.644306,-4.005748,6.224032,-5.629312,...,0.916134,-4.753392,-5.297943,3.110546,4.186899,3.808285,3.060312,-2.811014,-4.545404,2.232796
3,GSM3562837_3999840035_R02C02,3.935193,-4.879178,-5.518779,-3.143026,-1.970269,-6.487348,-3.460816,5.762759,-5.47399,...,0.241682,-4.635074,-4.702293,2.967417,3.834561,3.631687,3.381633,-3.322698,-4.960753,2.02318
4,GSM3562838_3999840035_R03C01,4.207299,-4.579331,-5.579593,-3.098945,-1.041133,-6.402856,-2.857366,5.775951,-5.572881,...,0.428073,-5.145249,-5.329275,2.833336,3.631742,3.819408,3.013578,-3.525631,-4.398742,2.936999


In [6]:
combined_df = mval.merge(phenotype_df, how='outer', on='ID')
combined_df.head()

,ID,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,...,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648,Sex,Age,Depression,Study,depr_one_hot
0,GSM3562834_3999840035_R01C01,3.699859,-4.60819,-5.758384,-3.428361,-1.607677,-6.670791,-3.107789,5.820279,-5.475742,...,3.712343,2.162832,-3.205565,-4.658252,1.941852,0,73,Control,1,0
1,GSM3562835_3999840035_R01C02,3.829019,-5.517733,-5.364797,-3.806508,-3.06947,-6.675133,-2.910615,6.075227,-5.647746,...,3.963833,3.127694,-3.446988,-4.824407,1.924064,1,32,Case,1,1
2,GSM3562836_3999840035_R02C01,4.543859,-5.298929,-5.672585,-3.913202,-2.032083,-6.644306,-4.005748,6.224032,-5.629312,...,3.808285,3.060312,-2.811014,-4.545404,2.232796,0,51,Case,1,1
3,GSM3562837_3999840035_R02C02,3.935193,-4.879178,-5.518779,-3.143026,-1.970269,-6.487348,-3.460816,5.762759,-5.47399,...,3.631687,3.381633,-3.322698,-4.960753,2.02318,1,30,Case,1,1
4,GSM3562838_3999840035_R03C01,4.207299,-4.579331,-5.579593,-3.098945,-1.041133,-6.402856,-2.857366,5.775951,-5.572881,...,3.819408,3.013578,-3.525631,-4.398742,2.936999,0,53,Case,1,1


In [7]:
combined_df.shape

(1630, 391216)

In [8]:
combined_df = combined_df.drop(columns=['ID','Depression'])

In [9]:
lr = LogisticRegression(max_iter = 10000)
rf = RandomForestClassifier()

In [10]:
x = combined_df.drop(columns=['Study','depr_one_hot'])
x = x.replace(-np.inf, pd.NA)
y = combined_df['depr_one_hot']
groups = combined_df['Study']

In [11]:
x = x.dropna(axis=1)

In [12]:
minmax_scaler = MinMaxScaler()
std_scaler = StandardScaler()
rob_scaler = RobustScaler()

In [13]:
x_minmax = minmax_scaler.fit_transform(x)

In [14]:
x_std = std_scaler.fit_transform(x)

In [15]:
x_rob = rob_scaler.fit_transform(x)

In [18]:
def split_train_pred(x, y, groups):
    sgkf = StratifiedGroupKFold(shuffle = True)
    lr_accu = []
    lr_auc = []
    rf_mean_accu = []
    rf_mean_auc = []
    for train_index, test_index in sgkf.split(x, y, groups):
        x_train_fold, x_test_fold = x[train_index], x[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]
        lr.fit(x_train_fold, y_train_fold)
        lr_pred = lr.predict(x_test_fold)
        lr_accu.append(accuracy_score(y_test_fold, lr_pred))
        lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
        rf_accu = 0
        rf_auc = 0
        for i in range(20):
            rf.fit(x_train_fold, y_train_fold)
            rf_pred = rf.predict(x_test_fold)
            rf_accu += accuracy_score(y_test_fold, rf_pred)
            rf_auc += roc_auc_score(y_test_fold, rf_pred)
        rf_mean_accu.append(rf_accu/20)
        rf_mean_auc.append(rf_auc/20)
    print('Stratified Group K Fold: \n')
    print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
    print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')
    sss = StratifiedShuffleSplit()
    lr_accu = []
    lr_auc = []
    rf_mean_accu = []
    rf_mean_auc = []
    for train_index, test_index in sss.split(x, y, groups):
        x_train_fold, x_test_fold = x[train_index], x[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]
        lr.fit(x_train_fold, y_train_fold)
        lr_pred = lr.predict(x_test_fold)
        lr_accu.append(accuracy_score(y_test_fold, lr_pred))
        lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
        rf_accu = 0
        rf_auc = 0
        for i in range(20):
            rf.fit(x_train_fold, y_train_fold)
            rf_pred = rf.predict(x_test_fold)
            rf_accu += accuracy_score(y_test_fold, rf_pred)
            rf_auc += roc_auc_score(y_test_fold, rf_pred)
        rf_mean_accu.append(rf_accu/20)
        rf_mean_auc.append(rf_auc/20)
    print('Stratifed Shuffle Split: \n')
    print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
    print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')
    skf = StratifiedKFold()
    lr_accu = []
    lr_auc = []
    rf_mean_accu = []
    rf_mean_auc = []
    for train_index, test_index in skf.split(x, y, groups):
        x_train_fold, x_test_fold = x[train_index], x[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]
        lr.fit(x_train_fold, y_train_fold)
        lr_pred = lr.predict(x_test_fold)
        lr_accu.append(accuracy_score(y_test_fold, lr_pred))
        lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
        rf_accu = 0
        rf_auc = 0
        for i in range(20):
            rf.fit(x_train_fold, y_train_fold)
            rf_pred = rf.predict(x_test_fold)
            rf_accu += accuracy_score(y_test_fold, rf_pred)
            rf_auc += roc_auc_score(y_test_fold, rf_pred)
        rf_mean_accu.append(rf_accu/20)
        rf_mean_auc.append(rf_auc/20)
    print('Stratified K Fold: \n')
    print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
    print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')
    gss = GroupShuffleSplit()
    lr_accu = []
    lr_auc = []
    rf_mean_accu = []
    rf_mean_auc = []
    for train_index, test_index in gss.split(x, y, groups):
        x_train_fold, x_test_fold = x[train_index], x[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]
        lr.fit(x_train_fold, y_train_fold)
        lr_pred = lr.predict(x_test_fold)
        lr_accu.append(accuracy_score(y_test_fold, lr_pred))
        lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
        rf_accu = 0
        rf_auc = 0
        for i in range(20):
            rf.fit(x_train_fold, y_train_fold)
            rf_pred = rf.predict(x_test_fold)
            rf_accu += accuracy_score(y_test_fold, rf_pred)
            rf_auc += roc_auc_score(y_test_fold, rf_pred)
        rf_mean_accu.append(rf_accu/20)
        rf_mean_auc.append(rf_auc/20)
    print('Group Shuffle Split: \n')
    print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
    print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')
    return

In [ ]:
print('Min Max Scaling: \n')
split_train_pred(x_minmax, y, groups)

Min Max Scaling: 



In [ ]:
print('Standard Scaling: \n')
split_train_pred(x_std, y, groups)

In [ ]:
print('\nRobust Scaling: \n')
split_train_pred(x_rob, y, groups)